# Experiment 10 – LSTM Based Autoencoder Using TensorFlow/Keras

### Deep Learning Laboratory

**Aim:** To implement an LSTM-based Autoencoder using TensorFlow/Keras for learning a compressed representation and reconstructing sequential data.

### Learning Objectives
- Understand the concept of an Autoencoder.
- Understand Encoder and Decoder architecture.
- Learn why LSTM is useful for sequential data.
- Build an LSTM Autoencoder using Keras.
- Train the model to reconstruct input sequences.
- Calculate reconstruction error and identify unusual sequences.

## 1. Theory

An **Autoencoder** is a neural network that learns to reproduce its input at the output. It has two main parts:

- **Encoder:** Converts the input sequence into a smaller hidden representation.
- **Decoder:** Uses the hidden representation to reconstruct the original sequence.

An **LSTM Autoencoder** uses LSTM layers in the encoder and decoder. LSTM is suitable for sequential data because it can learn information from previous time steps.

### Basic Architecture

```text
Input Sequence
      ↓
   LSTM Encoder
      ↓
Compressed Representation
      ↓
   LSTM Decoder
      ↓
Reconstructed Sequence
```

### Example

A sequence such as:

`[0.10, 0.20, 0.30, 0.40, 0.50]`

is given to the model. The autoencoder tries to produce a reconstructed sequence as close as possible to the original sequence.

The model is trained using **reconstruction error**, commonly Mean Squared Error (MSE).

In [ ]:
# Step 1: Import required libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

## 2. Generate Sequential Data

For a simple laboratory demonstration, we generate sine-wave sequences. Each sequence contains 50 time steps.

Using generated data makes the experiment reproducible and avoids downloading a large external dataset.

In [ ]:
# Step 2: Generate sine-wave sequences
np.random.seed(42)

N_SAMPLES = 2000
TIME_STEPS = 50

X = []

for _ in range(N_SAMPLES):
    start = np.random.uniform(0, 2 * np.pi)
    frequency = np.random.uniform(0.8, 1.2)
    t = np.linspace(start, start + 2 * np.pi * frequency, TIME_STEPS)
    sequence = np.sin(t)
    X.append(sequence)

X = np.array(X, dtype=np.float32)

# Add the feature dimension required by LSTM: (samples, time steps, features)
X = X[..., np.newaxis]

print("Dataset shape:", X.shape)

In [ ]:
# Step 3: Split into training and testing data
split = int(0.8 * len(X))

X_train = X[:split]
X_test = X[split:]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

In [ ]:
# Step 4: Display sample sequences
plt.figure(figsize=(10, 5))
for i in range(5):
    plt.plot(X_train[i].squeeze(), label=f"Sequence {i+1}")
plt.xlabel("Time Step")
plt.ylabel("Value")
plt.title("Sample Sequential Data")
plt.legend()
plt.show()

## 3. Build the LSTM Autoencoder

The model has:

1. **LSTM Encoder:** Compresses the sequence into a fixed-size representation.
2. **RepeatVector:** Repeats the compressed representation for every time step.
3. **LSTM Decoder:** Reconstructs the sequence.
4. **TimeDistributed Dense:** Produces one output value for each time step.


In [ ]:
# Step 5: Create the LSTM Autoencoder
LATENT_DIM = 16

model = keras.Sequential([
    layers.Input(shape=(TIME_STEPS, 1)),
    layers.LSTM(LATENT_DIM, activation="tanh"),
    layers.RepeatVector(TIME_STEPS),
    layers.LSTM(LATENT_DIM, activation="tanh", return_sequences=True),
    layers.TimeDistributed(layers.Dense(1))
])

model.compile(
    optimizer="adam",
    loss="mse"
)

model.summary()

## 4. Train the LSTM Autoencoder

The input and target are the same because an autoencoder learns to reconstruct its input.

```text
Input  →  Autoencoder  →  Output
  ↑                         ↓
  └──────── Same Data ──────┘
```


In [ ]:
# Step 6: Train the model
history = model.fit(
    X_train,
    X_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    verbose=1
)

In [ ]:
# Step 7: Evaluate reconstruction loss
train_loss = model.evaluate(X_train, X_train, verbose=0)
test_loss = model.evaluate(X_test, X_test, verbose=0)

print("Training Reconstruction MSE:", round(float(train_loss), 6))
print("Testing Reconstruction MSE:", round(float(test_loss), 6))

In [ ]:
# Step 8: Plot training and validation loss
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.title("LSTM Autoencoder Training")
plt.legend()
plt.show()

## 5. Compare Original and Reconstructed Sequences

The trained autoencoder receives test sequences and tries to reconstruct them. If learning is successful, the reconstructed curve should be close to the original curve.

In [ ]:
# Step 9: Reconstruct test sequences
reconstructed = model.predict(X_test, verbose=0)

plt.figure(figsize=(10, 6))
for i in range(3):
    plt.subplot(3, 1, i + 1)
    plt.plot(X_test[i].squeeze(), label="Original")
    plt.plot(reconstructed[i].squeeze(), label="Reconstructed")
    plt.ylabel("Value")
    plt.legend()
plt.xlabel("Time Step")
plt.tight_layout()
plt.show()

## 6. Calculate Reconstruction Error

Reconstruction error tells us how different the reconstructed sequence is from the original sequence.

```text
Reconstruction Error = Mean((Original − Reconstructed)²)
```

A smaller value means the autoencoder reconstructed the sequence more accurately.

In [ ]:
# Step 10: Calculate reconstruction error for each test sequence
errors = np.mean(np.square(X_test - reconstructed), axis=(1, 2))

print("Average reconstruction error:", round(float(np.mean(errors)), 6))
print("Minimum reconstruction error:", round(float(np.min(errors)), 6))
print("Maximum reconstruction error:", round(float(np.max(errors)), 6))

## 7. Optional: Detect an Unusual Sequence

An autoencoder trained mainly on normal patterns can produce a larger reconstruction error for an unusual pattern. This idea is often used for anomaly detection.

Here we create a noisy sequence and compare its reconstruction error with a normal sine-wave sequence.

In [ ]:
# Step 11: Create normal and unusual sequences
normal = X_test[0:1]

unusual = normal.copy()
noise = np.random.normal(0, 0.5, unusual.shape).astype(np.float32)
unusual = unusual + noise

normal_reconstruction = model.predict(normal, verbose=0)
unusual_reconstruction = model.predict(unusual, verbose=0)

normal_error = np.mean(np.square(normal - normal_reconstruction))
unusual_error = np.mean(np.square(unusual - unusual_reconstruction))

print("Normal sequence error:", round(float(normal_error), 6))
print("Unusual sequence error:", round(float(unusual_error), 6))

In [ ]:
# Step 12: Visualize normal and unusual sequences
plt.figure(figsize=(10, 6))

plt.subplot(2, 1, 1)
plt.plot(normal.squeeze(), label="Normal Original")
plt.plot(normal_reconstruction.squeeze(), label="Normal Reconstructed")
plt.title("Normal Sequence")
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(unusual.squeeze(), label="Unusual Input")
plt.plot(unusual_reconstruction.squeeze(), label="Unusual Reconstructed")
plt.title("Unusual Sequence")
plt.legend()

plt.tight_layout()
plt.show()

## 8. Student Practice

Try the following changes:

1. Change `LATENT_DIM` from `16` to `8` and compare reconstruction error.
2. Change it to `32` and compare the result.
3. Train for 10, 20 and 30 epochs.
4. Change the batch size from `32` to `64`.
5. Add more noise to the unusual sequence and observe the reconstruction error.

### Observation Table

| Configuration | Reconstruction Error |
|---|---|
| Latent dimension = 8 | ______ |
| Latent dimension = 16 | ______ |
| Latent dimension = 32 | ______ |

## Result

Thus, an **LSTM-based Autoencoder** was successfully implemented using TensorFlow/Keras. The model learned a compressed representation of sequential data and reconstructed the input sequences. Reconstruction error was also calculated to demonstrate how an autoencoder can be used for detecting unusual sequences.

## Viva Questions

1. What is an Autoencoder?
2. What are the two main parts of an Autoencoder?
3. What is an Encoder?
4. What is a Decoder?
5. Why is LSTM used in this experiment?
6. What is a latent representation?
7. What does `RepeatVector` do?
8. Why do we use `return_sequences=True` in the decoder?
9. What is reconstruction error?
10. How can an Autoencoder be used for anomaly detection?